> https://github.com/langchain-ai/langchain-mcp-adapters?tab=readme-ov-file#streamable-http

In [2]:
# Use server from examples/servers/streamable-http-stateless/

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

from langchain.agents import create_agent
from langchain_mcp_adapters.tools import load_mcp_tools

async with streamablehttp_client("http://127.0.0.1:8000/mcp/") as (read, write, _):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()

        # Get tools
        tools = await load_mcp_tools(session)

        result = await tools[0].ainvoke({"name": "김일남"})
        print(result)

[{'type': 'text', 'text': '안녕하세요, 김일남님!', 'id': 'lc_4cfccb59-138c-4db6-ba57-8f8ea9f16149'}]


In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "mcp_tools": {
            "url": "http://127.0.0.1:8000/mcp/",
            "transport": "streamable_http"
        }
    }
)

tools = await client.get_tools()

In [4]:
tools

[StructuredTool(name='hello', description='간단한 인사말을 반환하는 도구', args_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'helloArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000002C54C0ED4E0>),
 StructuredTool(name='get_current_time', description='현재 시각을 반환하는 함수\n\n    Args:\n        timezone (str): 타임존 (예: Asia/Seoul) 실제 존재하는 타임존이어야 함\n        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 LLM 답변 생성에 사용됨\n    ', args_schema={'properties': {'timezone': {'title': 'Timezone', 'type': 'string'}, 'location': {'title': 'Location', 'type': 'string'}}, 'required': ['timezone', 'location'], 'title': 'get_current_timeArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000002C54C0EE0C0>),
 StructuredTool(name='get_yf_stock_history', description='주식 종목의

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=tools,
    system_prompt="너는 사용자의 질문에 답변을 하기 위해 도구를 사용할 수 있다."
)

In [8]:
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "부산은 지금 몇시야?"}]}
)

In [9]:
result

{'messages': [HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}, id='e0d014ec-293e-48c0-9ef5-7b9e6755230d'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"timezone": "Asia/Seoul", "location": "\\ubd80\\uc0b0"}'}, '__gemini_function_call_thought_signatures__': {'218dd266-5701-4a90-bd53-8b33089dce54': 'CpoEAb4+9vswuI1DJZtkTGnFTQ12iYcWxBPY/KBLge8PBDABURsHI79cb79kIUyoAYGxSJ/Fe/oHeF/416BGN9ueawAylpGtXgfFDcDY/RPqDCSbgv9O8sXxmLfV0spXmrDjBEK5OSv0fQKAa3wePGD+Bggpl5ab6cPlbdwesYpLsdIapc8js5pPd6mb46Ucox71fFObMdfASdN3wzDglTKk1bBj/hFKcKeGDrkMRdqisNh8+e7eq2u1a1sbooveAWaodKzC0TGSsMJW83J7cONdKiXvFST99KQFk10Po1r/dpiv9cD4s1PIfD4ot9R7brATTaAdfIUv0VMh7K4YmTIMVi+ZeqTKmedzxGzfRLqkMd67sNMuJZo2dttijclu4NNQnVZCilMR+Dd6PjstcLCVNpbB7drke9MAm1j0zNUYkCVpd+4hJaX1FrZdIx7waYgPN85giedl/HdTqUcBr6YYJ2ZydbMXVWF4h1mBySSCWKI8UuiQ/EIb+97suaV7AFWV8QEdOG1z2BOexyIShVb5mhFyAgQMgzEJ6ljJosLeSbM44yvdjwyBwSquQddK3tjyBP2YbCyEyowxYr21QR/uV+h8ddYmJj

In [14]:
query = "두쫀쿠가 뭐야? 웹 검색 해봐."

In [15]:
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": query}]}
)

In [16]:
result

{'messages': [HumanMessage(content='두쫀쿠가 뭐야? 웹 검색 해봐.', additional_kwargs={}, response_metadata={}, id='60f5103d-3b2f-4d3a-81f7-13ef43a80417'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_web_search', 'arguments': '{"query": "\\ub450\\ucac0\\ucfe0"}'}, '__gemini_function_call_thought_signatures__': {'bfe61247-4ac9-4962-a082-b7c239f7f4ed': 'CroCAb4+9vtTiT9uHBaQGQ9dT9LzxlOK6aBBekpCAZmtQewEYCx5hzAcdEnwLtHNCeVIW1ZwdHbnwmhR2u4XvD7Z7U3Wdbd/1PvKA07jidieU64flnaom0wvmhschLcBxU0MsgriXxI5cv7Yg78ME8dNw6Ri4zeDm38EE3FJq966ZDM/P+jAPJ4KCE9QYZLeT+5USHSrboR8g9oe+teGQHv1G23WXig9k9EHrj5LjicAMscSDXDyTvvn/7D8L7u0egVB7I2XZ7gzxwNZFCicLsR0QV8gnc3dFry10VXoqIOLibpzc6Gbvf8lAO758rrvMLwPn6HFP9WuQGncFiBRpu0SIYDRLo9xC54A2ewKTxRZ+d7MTi/s+MKUOQGcbW+eQYmJhPT3fu06sNk4duw54vRwx0hKiiQr6UFFPVM='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c36d3-9db0-7b53-8b2f-a9fa34bcfcc7-0', tool_calls=